#  ⭐ `dim_concentracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root , normalize_concentracao

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet")  
bronze['CONCENTRACAO']= bronze['CONCENTRACAO'].fillna('DESCONHECIDO')
bronze = (
    bronze[['CONCENTRACAO']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['CONCENTRACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [ ]:
bronze.head()

In [4]:
bronze.CONCENTRACAO.nunique()

13230

In [5]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_CONCENTRACAO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [6]:
silver = bronze.copy()


In [7]:
silver.head(40)

,CONCENTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,460259
1,100mg,2452
2,500mg,2180
3,1g,1891
4,500 mg,1287
5,50mg,1237
6,100 mg,1201
7,10mg,1193
8,40mg,950
9,5 mg,916


In [8]:
silver = normalize_concentracao(silver, col="CONCENTRACAO")

In [9]:
silver.columns

Index(['CONCENTRACAO', 'FREQUENCIA_REGISTROS', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR'],
      dtype='object')

In [10]:
(
    silver.groupby("CONCENTRACAO_TIPO_CHAVE")["FREQUENCIA_REGISTROS"]
    .sum()
    .reset_index(name="FREQUENCIA_REGISTROS_TOTAL")
).sort_values(by='FREQUENCIA_REGISTROS_TOTAL', ascending=False)

,CONCENTRACAO_TIPO_CHAVE,FREQUENCIA_REGISTROS_TOTAL
0,desconhecido,467578
4,milligram (mg),43375
7,milligram per millilitre (mg/mL),29106
1,gram (g),6598
9,percent (%),2449
8,millilitre (mL),2058
3,microgram (ug),688
2,gram per millilitre (g/mL),626
6,milligram per milligram (mg/mg),351
5,milligram per gram (mg/g),58


In [11]:
hist =  silver[['CONCENTRACAO', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR']]


In [12]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_concentracao/hist_dim_concentracao.parquet", index=False)

# 🥇 Gold


In [15]:
dim = hist[['CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_TIPO_VALOR']].drop_duplicates().sort_values('CONCENTRACAO_TIPO_VALOR')
dim

,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR
0,desconhecido,0
1,milligram (mg),1
3,gram (g),2
197,microgram (ug),3
139,millilitre (mL),4
63,percent (%),5
13,milligram per millilitre (mg/mL),6
679,milligram per milligram (mg/mg),7
1393,milligram per gram (mg/g),8
149,gram per millilitre (g/mL),9


In [16]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_concentracao/dim_concentracao.csv", sep=";", index=False)